# Compressed-LLM Pilot on Kaggle

Before running: attach this project as a Kaggle Dataset, enable Internet, and set the notebook Accelerator to GPU.

In [ ]:
from pathlib import Path

candidates = sorted(Path('/kaggle/input').glob('**/scripts/kaggle_bootstrap.py'))
if not candidates:
    raise FileNotFoundError('Attach the project as a Kaggle Dataset, or clone it into /kaggle/working.')
BOOTSTRAP = candidates[0]
BOOTSTRAP

In [ ]:
!python {BOOTSTRAP} --install
%cd /kaggle/working/compression-eval

## Smoke test

This verifies the pipeline. The numbers are meaningless because the model and sample size are tiny.

In [ ]:
!python -m pilot_eval.run --config configs/kaggle_smoke_tiny.yaml
!python -m pilot_eval.analyze --run-dir /kaggle/working/results/kaggle_smoke_tiny --baseline fp16 --bootstrap 100

## Optional: install quantization backends

Run this before building GPTQ/AWQ checkpoints on Kaggle.

In [ ]:
!python scripts/kaggle_bootstrap.py --install --with-quantization

## Build one quantized checkpoint

Run this cell repeatedly, changing `METHOD` and `SEED`. Save a Kaggle version after successful builds.

In [ ]:
METHOD = 'gptq'  # 'gptq' or 'awq'
SEED = 0
out = f'/kaggle/working/outputs/quantized/qwen25-1p5b-{METHOD}4-seed{SEED}'

!python scripts/build_quantized.py \
  --model-id Qwen/Qwen2.5-1.5B-Instruct \
  --method {METHOD} \
  --seed {SEED} \
  --output-dir {out} \
  --trust-remote-code

## Evaluate the pilot

Run after all six local quantized checkpoints exist: GPTQ/AWQ seeds 0, 1, and 2.

In [ ]:
!python -m pilot_eval.run --config configs/kaggle_qwen_1p5b.yaml
!python -m pilot_eval.analyze --run-dir /kaggle/working/results/kaggle_qwen25_1p5b_pilot --baseline fp16 --bootstrap 2000

## Package results

In [ ]:
!python scripts/kaggle_pack_outputs.py